# MITS: Synthetic Dialog Generation

Generate Socratic tutoring dialogs for fine-tuning GLM-4.7-Flash.

**Matrix**: 5 topics × 4 difficulties × 5 student types = 100 combinations

**Output**: `data/training/synthetic_dialogs.jsonl`

In [ ]:
# Install dependencies (Colab)
# !pip install ollama pydantic-settings structlog

In [ ]:
import json
import uuid
import random
import time
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from enum import Enum

# For Colab with Ollama or API-based generation
try:
    import ollama
    HAS_OLLAMA = True
except ImportError:
    HAS_OLLAMA = False
    print("ollama not available, will use API mode")

## 1. Configuration

In [ ]:
# Generation config
MODEL = "glm4:9b"  # or "glm-4.7-flash" for Ollama, or API endpoint
DIALOGS_PER_COMBINATION = 10  # 100 combinations × 10 = 1000 dialogs
MIN_TURNS = 6
MAX_TURNS = 16
OUTPUT_DIR = Path("data/training")
OUTPUT_FILE = OUTPUT_DIR / "synthetic_dialogs.jsonl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Topics (5)
TOPICS = {
    "derivatives": {
        "name_ru": "Производные",
        "skills": ["правило степени", "правило произведения", "правило частного", "цепное правило", "производные тригонометрических функций"]
    },
    "integrals": {
        "name_ru": "Интегралы",
        "skills": ["неопределённый интеграл", "определённый интеграл", "замена переменной", "интегрирование по частям", "площадь под кривой"]
    },
    "limits": {
        "name_ru": "Пределы",
        "skills": ["предел функции", "односторонние пределы", "раскрытие неопределённостей", "правило Лопиталя", "замечательные пределы"]
    },
    "equations": {
        "name_ru": "Уравнения",
        "skills": ["линейные уравнения", "квадратные уравнения", "системы уравнений", "тригонометрические уравнения", "показательные уравнения"]
    },
    "linear_algebra": {
        "name_ru": "Линейная алгебра",
        "skills": ["матрицы", "определители", "собственные значения", "линейные преобразования", "системы линейных уравнений"]
    }
}

# Difficulties (4)
DIFFICULTIES = ["easy", "medium", "hard", "olympiad"]

# Student types (5)
STUDENT_TYPES = ["novice", "intermediate", "advanced", "confused", "curious"]

print(f"Combinations: {len(TOPICS)} × {len(DIFFICULTIES)} × {len(STUDENT_TYPES)} = {len(TOPICS) * len(DIFFICULTIES) * len(STUDENT_TYPES)}")
print(f"Total dialogs: {len(TOPICS) * len(DIFFICULTIES) * len(STUDENT_TYPES) * DIALOGS_PER_COMBINATION}")

## 2. Prompt Templates

In [ ]:
SYSTEM_PROMPT = """Ты — сократический репетитор по математике. Твоя миссия — помогать студентам самостоятельно находить решения через наводящие вопросы.

ПРИНЦИПЫ:
1. НИКОГДА не давай готовых ответов сразу
2. Задавай ОДИН чёткий вопрос за раз
3. Используй $LaTeX$ для математики: $x^2$, $\\frac{a}{b}$, $\\sqrt{x}$
4. Отвечай на русском языке

ТИПЫ ХОДОВ (move):
- scaffolding: Разбить на шаги, направить вопросом
- problematize: Спросить \"почему?\" или \"что если?\"
- rectify: Мягко указать на ошибку
- encourage: Похвалить прогресс
- hint: Дать подсказку (если студент застрял)
- tell: Раскрыть ответ (ТОЛЬКО в крайнем случае!)

ФОРМАТ ОТВЕТА (JSON):
{\"move\": \"scaffolding\", \"message\": \"Ответ с $LaTeX$\", \"reasoning\": \"Обоснование\"}"""


TASK_GENERATION_PROMPT = """Сгенерируй математическую задачу:
- Тема: {topic} ({topic_ru})
- Сложность: {difficulty}
- Навыки: {skills}

Верни JSON:
{{
    "problem": "Условие задачи с $LaTeX$",
    "solution": "Пошаговое решение с $LaTeX$",
    "answer": "Краткий ответ",
    "hints": ["Подсказка 1", "Подсказка 2", "Подсказка 3"],
    "common_mistakes": ["Типичная ошибка 1", "Типичная ошибка 2"]
}}

Сложность определяется:
- easy: Базовая задача, 1-2 шага
- medium: Стандартная задача, 3-4 шага
- hard: Сложная задача, 5+ шагов, комбинация навыков
- olympiad: Олимпиадная задача, нестандартный подход"""


STUDENT_PERSONA_PROMPTS = {
    "novice": "Ты студент-новичок. Не понимаешь базовые концепции, делаешь простые ошибки, часто говоришь 'не понимаю'. Отвечай 1-2 предложениями.",
    "intermediate": "Ты студент среднего уровня. Частично понимаешь тему, ошибаешься в деталях. Отвечай 1-3 предложениями.",
    "advanced": "Ты продвинутый студент. Хорошо понимаешь тему, делаешь тонкие ошибки, задаёшь умные вопросы. Отвечай 1-3 предложениями.",
    "confused": "Ты растерянный студент. Расстроен, просишь помощи, говоришь 'запутался', 'не получается'. Отвечай 1-2 предложениями.",
    "curious": "Ты любопытный студент. Спрашиваешь 'почему?', 'а что если?', хочешь понять глубже. Отвечай 1-3 предложениями."
}


DIALOG_SIMULATION_PROMPT = """Сгенерируй полный диалог между репетитором и студентом.

ЗАДАЧА:
{task_json}

ТИП СТУДЕНТА: {student_type}
Характеристика: {student_desc}

ТРЕБОВАНИЯ:
- Минимум {min_turns} реплик (суммарно студент+репетитор)
- Максимум {max_turns} реплик
- Репетитор использует РАЗНЫЕ типы ходов (scaffolding, problematize, rectify, encourage, hint)
- Студент постепенно приближается к решению
- Используй $LaTeX$ для математических формул
- Весь диалог на русском языке

Верни JSON массив реплик:
[
    {{"role": "student", "content": "..."}},
    {{"role": "tutor", "content": "...", "move": "scaffolding", "reasoning": "..."}},
    ...
]

ВАЖНО: Верни ТОЛЬКО JSON массив, без пояснений."""

## 3. LLM Client

In [ ]:
def generate_llm(prompt: str, system: str = "", json_mode: bool = False, temperature: float = 0.7) -> str:
    """Generate text using Ollama."""
    if not HAS_OLLAMA:
        raise RuntimeError("Ollama not available")
    
    options = {"temperature": temperature}
    
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": system} if system else None,
            {"role": "user", "content": prompt}
        ],
        format="json" if json_mode else "",
        options=options
    )
    
    return response["message"]["content"]


def parse_json_response(text: str) -> dict | list:
    """Parse JSON from LLM response, handling markdown code blocks."""
    text = text.strip()
    # Remove markdown code fences
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
    return json.loads(text)

## 4. Task Generation

In [ ]:
def generate_task(topic: str, difficulty: str) -> dict:
    """Generate a math task for the given topic and difficulty."""
    topic_info = TOPICS[topic]
    prompt = TASK_GENERATION_PROMPT.format(
        topic=topic,
        topic_ru=topic_info["name_ru"],
        difficulty=difficulty,
        skills=", ".join(topic_info["skills"])
    )
    
    response = generate_llm(prompt, json_mode=True, temperature=0.8)
    task = parse_json_response(response)
    task["topic"] = topic
    task["difficulty"] = difficulty
    task["id"] = str(uuid.uuid4())
    return task


# Test task generation
if HAS_OLLAMA:
    test_task = generate_task("derivatives", "medium")
    print(json.dumps(test_task, ensure_ascii=False, indent=2))
else:
    print("Skipping test - Ollama not available")

## 5. Dialog Generation

In [ ]:
def generate_dialog(task: dict, student_type: str) -> dict:
    """Generate a full tutoring dialog for a task and student type."""
    prompt = DIALOG_SIMULATION_PROMPT.format(
        task_json=json.dumps(task, ensure_ascii=False, indent=2),
        student_type=student_type,
        student_desc=STUDENT_PERSONA_PROMPTS[student_type],
        min_turns=MIN_TURNS,
        max_turns=MAX_TURNS
    )
    
    response = generate_llm(prompt, system=SYSTEM_PROMPT, json_mode=True, temperature=0.7)
    turns = parse_json_response(response)
    
    if not isinstance(turns, list):
        raise ValueError(f"Expected list, got {type(turns)}")
    
    # Build dialog object
    dialog = {
        "id": str(uuid.uuid4()),
        "task": task,
        "student_persona": student_type,
        "turns": turns,
        "metadata": {
            "num_turns": len(turns),
            "model": MODEL,
            "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S")
        }
    }
    
    # Analyze outcomes
    tutor_turns = [t for t in turns if t.get("role") == "tutor"]
    dialog["metadata"]["solved"] = any(
        t.get("move") == "encourage" and "правильно" in t.get("content", "").lower()
        for t in tutor_turns
    )
    dialog["metadata"]["told_answer"] = any(
        t.get("move") == "tell" for t in tutor_turns
    )
    dialog["metadata"]["moves_used"] = list(set(
        t.get("move", "") for t in tutor_turns if t.get("move")
    ))
    
    return dialog


# Test dialog generation
if HAS_OLLAMA:
    test_dialog = generate_dialog(test_task, "novice")
    print(f"Turns: {test_dialog['metadata']['num_turns']}")
    print(f"Moves: {test_dialog['metadata']['moves_used']}")
    for t in test_dialog["turns"][:4]:
        role = t["role"]
        move = f" [{t.get('move', '')}]" if t.get('move') else ""
        print(f"  {role}{move}: {t['content'][:80]}...")

## 6. Batch Generation

In [ ]:
def generate_batch(
    dialogs_per_combination: int = DIALOGS_PER_COMBINATION,
    output_file: Path = OUTPUT_FILE,
    resume: bool = True
):
    """Generate all dialogs across the full matrix."""
    # Resume support: load existing dialogs
    existing_ids = set()
    existing_combos = set()
    if resume and output_file.exists():
        with open(output_file, "r", encoding="utf-8") as f:
            for line in f:
                d = json.loads(line)
                existing_ids.add(d["id"])
                combo = (d["task"]["topic"], d["task"]["difficulty"], d["student_persona"])
                existing_combos.add(combo)
        print(f"Resuming: {len(existing_ids)} existing dialogs")
    
    total_combinations = len(TOPICS) * len(DIFFICULTIES) * len(STUDENT_TYPES)
    total_target = total_combinations * dialogs_per_combination
    generated = len(existing_ids)
    errors = 0
    
    with open(output_file, "a" if resume else "w", encoding="utf-8") as f:
        for topic in TOPICS:
            for difficulty in DIFFICULTIES:
                # Generate task once per topic+difficulty
                try:
                    task = generate_task(topic, difficulty)
                except Exception as e:
                    print(f"  ERROR generating task {topic}/{difficulty}: {e}")
                    errors += 1
                    continue
                
                for student_type in STUDENT_TYPES:
                    for i in range(dialogs_per_combination):
                        combo = (topic, difficulty, student_type)
                        
                        try:
                            dialog = generate_dialog(task, student_type)
                            f.write(json.dumps(dialog, ensure_ascii=False) + "\n")
                            f.flush()
                            generated += 1
                            
                            if generated % 10 == 0:
                                print(f"  Progress: {generated}/{total_target} ({100*generated/total_target:.1f}%) | Errors: {errors}")
                        
                        except Exception as e:
                            print(f"  ERROR {topic}/{difficulty}/{student_type}: {e}")
                            errors += 1
                            continue
    
    print(f"\nDone! Generated: {generated}, Errors: {errors}")
    print(f"Output: {output_file}")
    return generated, errors

In [ ]:
# Run batch generation (uncomment to execute)
# generated, errors = generate_batch(dialogs_per_combination=10)

## 7. Dataset Statistics

In [ ]:
def analyze_dataset(path: Path = OUTPUT_FILE):
    """Analyze generated dialog dataset."""
    if not path.exists():
        print(f"File not found: {path}")
        return
    
    dialogs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            dialogs.append(json.loads(line))
    
    print(f"Total dialogs: {len(dialogs)}")
    print(f"Total turns: {sum(len(d['turns']) for d in dialogs)}")
    print(f"Avg turns/dialog: {sum(len(d['turns']) for d in dialogs) / len(dialogs):.1f}")
    
    # Topic distribution
    from collections import Counter
    topics = Counter(d["task"]["topic"] for d in dialogs)
    print(f"\nTopics: {dict(topics)}")
    
    difficulties = Counter(d["task"]["difficulty"] for d in dialogs)
    print(f"Difficulties: {dict(difficulties)}")
    
    personas = Counter(d["student_persona"] for d in dialogs)
    print(f"Personas: {dict(personas)}")
    
    # Move distribution
    all_moves = []
    for d in dialogs:
        for t in d["turns"]:
            if t.get("move"):
                all_moves.append(t["move"])
    moves = Counter(all_moves)
    total_moves = sum(moves.values())
    print(f"\nMove distribution:")
    for move, count in moves.most_common():
        print(f"  {move}: {count} ({100*count/total_moves:.1f}%)")
    
    # Outcomes
    solved = sum(1 for d in dialogs if d.get("metadata", {}).get("solved", False))
    told = sum(1 for d in dialogs if d.get("metadata", {}).get("told_answer", False))
    print(f"\nSolved: {solved}/{len(dialogs)} ({100*solved/len(dialogs):.1f}%)")
    print(f"Told answer: {told}/{len(dialogs)} ({100*told/len(dialogs):.1f}%)")


analyze_dataset()